## Screen by bond valance

In [1]:
from pymatgen.analysis.bond_valence import BVAnalyzer

In [14]:
import mp_api
import pandas as pd
from mp_offline.client import MPOffline, MaterialSummary
from pymatgen.core.composition import Composition
from pymatgen.core import Structure
from tqdm.auto import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter
from monty.serialization import dumpfn
from copy import deepcopy

## Loading from the MP dataset

In [4]:
client = MPOffline()
objs = client.query_all(MaterialSummary.energy_above_hull < 0.01)

In [5]:
records = []
for (obj, ) in objs:
    record = {}
    structure = Structure.from_dict(obj.structure)
    record['composition'] = structure.composition
    record['# comp'] = structure.composition
    record['gap'] = obj.json_data['band_gap']
    record['band_gap'] = obj.json_data['band_gap']
    record['nelems'] = len(record['composition'])
    record['reduced_composition'] = record['composition'].get_reduced_composition_and_factor()[0]
    record['# comp'] =  record['reduced_composition']
    record['e_form'] = obj.formation_energy_per_atom
    record['e_hull'] = obj.energy_above_hull
    record['vol'] = structure.volume
    record['nsites'] = obj.nsites
    record['reduced_form'] = structure.composition.reduced_formula
    record['material_id'] = obj.material_id
    record['structure'] = structure
    record['e'] = obj.nsites * obj.energy_per_atom
    records.append(record)
df_mp = pd.DataFrame.from_records(records).set_index('material_id')

/home/bonan/miniconda3/envs/work/lib/python3.10/site-packages/pymatgen/core/composition.py:1337: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])
/home/bonan/miniconda3/envs/work/lib/python3.10/site-packages/pymatgen/core/composition.py:1337: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])
/home/bonan/miniconda3/envs/work/lib/python3.10/site-packages/pymatgen/core/composition.py:1337: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])


In [6]:
from importlib import reload
import datacollect
reload(datacollect)

# df_all = datacollect.load_wbm_data('wbm-dataset', 'wbm-dataset')

<module 'datacollect' from '/home/bonan/work/HC/pair-screening/datacollect.py'>

Select binary structure that are close to the convex hull and have small PBE band gap

In [7]:
from pickle import load

with open('wbm-dataset.df', 'rb') as fp:
    df_wbm = load(fp)

## Locate structure that valence can be assigned

If a structure cannot be assigned with a valence it is probably metallic (or it can be mixed valence).  
These kind of metallic structure will not be able to for solid solutions with tunnable properties, so we can discard them in the very begining!

In [8]:
ana = BVAnalyzer()

In [15]:
valid_ids_mp = []
for mp_id, row in tqdm(df_mp.iterrows(), total=len(df_mp)):
    try:
        ana.get_valences(row.structure)
    except ValueError:
        continue
    valid_ids_mp.append(mp_id)
dumpfn(valid_ids_mp, 'mp_id_with_valence.json')

  0%|          | 0/46675 [00:00<?, ?it/s]

spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.


In [16]:
valid_ids_wbm = []
for mp_id, row in tqdm(df_wbm.iterrows(), total=len(df_wbm)):
    try:
        ana.get_valences(row.structure)
    except ValueError:
        continue
    valid_ids_wbm.append(mp_id)
dumpfn(valid_ids_wbm, 'wbm_id_with_valence.json')

  0%|          | 0/257489 [00:00<?, ?it/s]

spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
